# Can prior search signals flag next-month content decline across new clients?

**ML-11 Capstone | Author: Mahdia Faheem | FlyRank warehouse build `v20260703`**

## Abstract

This paper asks whether recent Google Search Console signals can help an editor prioritize content pages at risk of a sharp next-month impression decline. It uses the full pseudonymized FlyRank warehouse daily-performance table, aggregating observations to client-content-month and filtering for adequate measurement coverage. A fixed Random Forest is compared with a transparent recent-trend baseline using features from the two completed months before each target month, with June 2026 and a client holdout reserved for the final test. **Headline test result: [replace with the executed results and test base rate].** The result is retrospective decision support for human review, not a causal estimate of refresh impact or a guarantee of future performance.

> Do not submit while the result placeholder remains. The abstract and paper must match the executed full-release results below.

## 1. Question - problem framing

**Research question:** Can the last two complete months of search-performance signals rank content items whose average daily Google Search Console impressions will fall by more than 20% in the following month?

**Decision supported:** Which existing content-client pairs should an editor investigate first for a possible technical, seasonal, search-intent, or content issue? The model ranks review priority; it does not prescribe an edit.

**Unit of analysis:** One pseudonymized content item for one client and one target month. A false positive spends editorial time on an item that does not cross the decline threshold; a false negative may delay investigation of a real decline. Because the cost of a wrong content change can exceed the cost of a review, every recommendation requires a human check.

In [12]:
research_question = (
    "Can the previous two complete months of search signals rank content items that will "
    "have a >20% fall in average daily impressions in the following month?"
)
print("Research question:", research_question)
print("Decision: prioritize a human investigation, not an automatic content change.")
print("Unit: one pseudonymous content item x client x target month.")

Research question: Can the previous two complete months of search signals rank content items that will have a >20% fall in average daily impressions in the following month?
Decision: prioritize a human investigation, not an automatic content change.
Unit: one pseudonymous content item x client x target month.


## 2. Data - source, windows, and exclusions

This analysis uses the gated **FlyRank/internship-warehouse** release, build `flyrank_pseudonymized_warehouse_release_v20260703`, exported 2026-07-03. The daily fact table contains 78,835,655 rows at `report_date x client x content` grain, spanning 2025-01-27 through 2026-06-30. The final analysis reads `fact_content_daily_performance`; the sample table was used only for schema/coverage checks, not for reported model metrics.

The analysis keeps GSC impressions, clicks, average position, and the `gsc_data_available` flag. It aggregates them to calendar-month average-per-available-day rates. A month needs at least 20 available GSC days; the preceding two months together need at least 200 impressions. These filters reduce conclusions based on sparse observations and tiny denominators.

The warehouse's fixed 90-day query table is excluded because its window overlaps the outcome period. GA4 signals are excluded because availability differs across clients and the question is specifically about search impressions. Client/content hashes are used only for grouping and the held-out-client split, never as model features. No names, URLs, titles, keywords, or raw queries are used or printed.

In [13]:
from pathlib import Path
import getpass
import os

import duckdb
import numpy as np
import pandas as pd

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "scripts" / "run_all.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing scripts/run_all.py.")

work_outputs = repo_root / "work" / "outputs"
work_outputs.mkdir(parents=True, exist_ok=True)
feature_cache = work_outputs / "capstone_feature_panel_v1.parquet"
release_root = "hf://datasets/FlyRank/internship-warehouse"
fact_source = f"{release_root}/fact_content_daily_performance/**/*.parquet"

feature_sql = f"""
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', report_date)::DATE AS month,
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions,
        SUM(CASE WHEN gsc_data_available THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks,
        AVG(gsc_avg_position) FILTER (
            WHERE gsc_data_available AND gsc_avg_position > 0
        ) AS avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_data_available) AS gsc_days
    FROM read_parquet('{fact_source}')
    WHERE report_date BETWEEN DATE '2025-01-27' AND DATE '2026-06-30'
    GROUP BY 1, 2, 3
), windowed AS (
    SELECT *,
        LAG(month) OVER entity_month AS previous_month,
        LAG(impressions) OVER entity_month AS previous_impressions,
        LAG(clicks) OVER entity_month AS previous_clicks,
        LAG(avg_position) OVER entity_month AS previous_avg_position,
        LAG(gsc_days) OVER entity_month AS previous_gsc_days,
        LEAD(month) OVER entity_month AS target_month,
        LEAD(impressions) OVER entity_month AS target_impressions,
        LEAD(gsc_days) OVER entity_month AS target_gsc_days
    FROM monthly
    WINDOW entity_month AS (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY month
    )
), rates AS (
    SELECT *,
        impressions / NULLIF(gsc_days, 0) AS current_impressions_per_gsc_day,
        previous_impressions / NULLIF(previous_gsc_days, 0) AS previous_impressions_per_gsc_day,
        clicks / NULLIF(gsc_days, 0) AS current_clicks_per_gsc_day,
        previous_clicks / NULLIF(previous_gsc_days, 0) AS previous_clicks_per_gsc_day,
        target_impressions / NULLIF(target_gsc_days, 0) AS target_impressions_per_gsc_day
    FROM windowed
)
SELECT
    client_hash_id,
    content_hash_id,
    month AS feature_month,
    target_month,
    current_impressions_per_gsc_day,
    previous_impressions_per_gsc_day,
    current_clicks_per_gsc_day,
    previous_clicks_per_gsc_day,
    avg_position AS current_avg_position,
    previous_avg_position,
    CASE
        WHEN target_impressions_per_gsc_day <
             0.8 * ((current_impressions_per_gsc_day + previous_impressions_per_gsc_day) / 2.0)
        THEN 1 ELSE 0
    END AS is_declining
FROM rates
WHERE previous_month = DATE_TRUNC('month', month - INTERVAL '1 month')::DATE
  AND target_month = DATE_TRUNC('month', month + INTERVAL '1 month')::DATE
  AND gsc_days >= 20
  AND previous_gsc_days >= 20
  AND target_gsc_days >= 20
  AND impressions + previous_impressions >= 200
"""

if feature_cache.is_file():
    panel = duckdb.read_parquet(str(feature_cache)).df()
    print("Using the local, ignored feature cache; the warehouse fact table was not rescanned.")
else:
    from huggingface_hub import get_token

    hf_token = os.environ.get("HF_TOKEN") or get_token()
    if not hf_token:
        hf_token = getpass.getpass("Enter your Hugging Face READ token (input is hidden): ")
    if not hf_token:
        raise ValueError("Hugging Face authentication is required to read the gated warehouse.")

    con = duckdb.connect()
    token_sql = hf_token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token_sql}')")
    del hf_token, token_sql

    temporary_cache = feature_cache.with_name("capstone_feature_panel_v1.tmp.parquet")
    if temporary_cache.exists():
        temporary_cache.unlink()
    cache_sql_path = temporary_cache.as_posix().replace("'", "''")
    con.execute(
        f"COPY ({feature_sql}) TO '{cache_sql_path}' "
        "(FORMAT PARQUET, COMPRESSION ZSTD)"
    )
    temporary_cache.replace(feature_cache)
    con.close()
    panel = pd.read_parquet(feature_cache)

required_columns = {
    "client_hash_id", "content_hash_id", "feature_month", "target_month",
    "current_impressions_per_gsc_day", "previous_impressions_per_gsc_day",
    "current_clicks_per_gsc_day", "previous_clicks_per_gsc_day",
    "current_avg_position", "previous_avg_position", "is_declining",
}
missing_columns = sorted(required_columns - set(panel.columns))
if missing_columns:
    raise ValueError(f"The cached feature panel is missing required columns: {missing_columns}")
if panel.empty:
    raise ValueError("No eligible page-month examples were produced; inspect coverage filters and source access.")

panel["feature_month"] = pd.to_datetime(panel["feature_month"])
panel["target_month"] = pd.to_datetime(panel["target_month"])
panel["is_declining"] = panel["is_declining"].astype(int)
print("Release: FlyRank/internship-warehouse, build v20260703")
print("Source table: fact_content_daily_performance; published row count: 78,835,655")
print(f"Eligible page-month examples: {len(panel):,}")
print(
    "Target months:",
    panel["target_month"].min().strftime("%Y-%m"),
    "through",
    panel["target_month"].max().strftime("%Y-%m"),
)
print("The cached Parquet contains pseudonymous join keys and is gitignored; do not commit it.")

Using the local, ignored feature cache; the warehouse fact table was not rescanned.
Release: FlyRank/internship-warehouse, build v20260703
Source table: fact_content_daily_performance; published row count: 78,835,655
Eligible page-month examples: 469,410
Target months: 2025-04 through 2026-06
The cached Parquet contains pseudonymous join keys and is gitignored; do not commit it.


## 3. Methodology - a future-month and client holdout

For each client-content pair, the features use two consecutive completed calendar months; the label is defined on the immediately following month. The label is 1 when the outcome month's average daily impressions are below 80% of the mean daily impressions in the two feature months. Every included month has at least 20 GSC-available days, and the two prior months have at least 200 impressions combined.

The transparent baseline ranks pages by the most recent historical impression-rate drop. The pre-specified model is a Random Forest using log impressions/clicks rates, average positions, position change, historical impression change, and historical CTR from those same prior months. `client_hash_id`, `content_hash_id`, target-month metrics, `trend_direction`, and `trend_pct` are not features.

The test is June 2026, the latest complete target month. A seeded 20% of clients with eligible June examples are held out entirely; training uses earlier target months from the other clients. This tests future-month ranking for clients not used in training. The seed is 42. The held-out group is chosen before model fitting, and the Random Forest settings are fixed rather than tuned on the test.

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

SEED = 42
TEST_TARGET_MONTH = pd.Timestamp("2026-06-01")
FEATURE_COLUMNS = [
    "log_current_impressions",
    "log_previous_impressions",
    "log_current_clicks",
    "log_previous_clicks",
    "current_avg_position",
    "previous_avg_position",
    "position_change",
    "historical_impression_change",
    "current_ctr",
    "previous_ctr",
]

panel["log_current_impressions"] = np.log1p(
    panel["current_impressions_per_gsc_day"].clip(lower=0)
)
panel["log_previous_impressions"] = np.log1p(
    panel["previous_impressions_per_gsc_day"].clip(lower=0)
)
panel["log_current_clicks"] = np.log1p(
    panel["current_clicks_per_gsc_day"].clip(lower=0)
)
panel["log_previous_clicks"] = np.log1p(
    panel["previous_clicks_per_gsc_day"].clip(lower=0)
)
panel["position_change"] = panel["current_avg_position"] - panel["previous_avg_position"]
previous_rate = panel["previous_impressions_per_gsc_day"].clip(lower=1.0)
panel["historical_impression_change"] = (
    (panel["current_impressions_per_gsc_day"] - panel["previous_impressions_per_gsc_day"])
    / previous_rate
).clip(-5, 5)
panel["current_ctr"] = (
    panel["current_clicks_per_gsc_day"]
    / panel["current_impressions_per_gsc_day"].replace(0, np.nan)
)
panel["previous_ctr"] = (
    panel["previous_clicks_per_gsc_day"]
    / panel["previous_impressions_per_gsc_day"].replace(0, np.nan)
)
panel[FEATURE_COLUMNS] = panel[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)

june_examples = panel.loc[panel["target_month"].eq(TEST_TARGET_MONTH)].copy()
if june_examples.empty:
    raise ValueError("The June 2026 target month has no eligible examples.")
client_ids = np.sort(june_examples["client_hash_id"].unique())
if len(client_ids) < 5:
    raise ValueError("Fewer than five clients have eligible June examples for a stable group holdout.")
test_client_count = max(1, int(np.ceil(0.20 * len(client_ids))))
test_clients = set(np.random.default_rng(SEED).permutation(client_ids)[:test_client_count])

train_df = panel.loc[
    panel["target_month"].lt(TEST_TARGET_MONTH)
    & ~panel["client_hash_id"].isin(test_clients)
].copy()
test_df = june_examples.loc[june_examples["client_hash_id"].isin(test_clients)].copy()
if train_df.empty or test_df.empty:
    raise ValueError("The chronological client holdout produced an empty train or test set.")
if train_df["is_declining"].nunique() != 2 or test_df["is_declining"].nunique() != 2:
    raise ValueError("Both train and test must contain both decline classes; do not tune on test outcomes.")

model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                min_samples_leaf=50,
                class_weight="balanced_subsample",
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    ]
)
model.fit(train_df[FEATURE_COLUMNS], train_df["is_declining"])
model_scores = model.predict_proba(test_df[FEATURE_COLUMNS])[:, 1]
test_scored = test_df[
    [
        "client_hash_id", "content_hash_id", "feature_month", "target_month",
        "is_declining", "historical_impression_change", "position_change",
        "current_ctr", "previous_ctr",
    ]
].copy()
test_scored["model_score"] = model_scores
test_scored["baseline_score"] = -test_scored["historical_impression_change"]

train_clients = set(train_df["client_hash_id"].unique())
assert train_clients.isdisjoint(test_clients)
print(f"Training examples: {len(train_df):,}; training clients: {len(train_clients):,}")
print(f"Test examples: {len(test_df):,}; held-out clients: {test_client_count:,}")
print(
    "Training target months:",
    train_df["target_month"].min().strftime("%Y-%m"),
    "through",
    train_df["target_month"].max().strftime("%Y-%m"),
)
print("Test target month: 2026-06; test labels are evaluated only after scoring.")

Training examples: 350,172; training clients: 30
Test examples: 4,245; held-out clients: 9
Training target months: 2025-05 through 2026-05
Test target month: 2026-06; test labels are evaluated only after scoring.


## 4. Results - model vs. baseline on the same test

Both ranking methods are evaluated on the same June 2026 page-month examples from the same held-out clients. Precision@20 and precision@50 are shown beside the test-set decline base rate; ROC AUC and average precision summarize ranking quality over the full test set. The test month and client group are not used to fit the model.

In [15]:
from sklearn.metrics import average_precision_score, roc_auc_score

y_test = test_scored["is_declining"].to_numpy(dtype=int)
test_base_rate = float(y_test.mean())
if len(np.unique(y_test)) != 2:
    raise ValueError("ROC AUC and average precision require both classes in the test set.")

def precision_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")
    selected = order[: min(k, len(order))]
    return float(labels[selected].mean())

def ranking_metrics(scores):
    return {
        "base_rate": test_base_rate,
        "roc_auc": float(roc_auc_score(y_test, scores)),
        "average_precision": float(average_precision_score(y_test, scores)),
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "precision_at_50": precision_at_k(y_test, scores, 50),
        "precision_at_100": precision_at_k(y_test, scores, 100),
    }

baseline_metrics = ranking_metrics(test_scored["baseline_score"])
model_metrics = ranking_metrics(test_scored["model_score"])
results_table = pd.DataFrame(
    [
        {"Ranker": "Recent-trend baseline", **baseline_metrics},
        {"Ranker": "Random Forest", **model_metrics},
    ]
)
print(results_table.to_string(index=False, formatters={
    "base_rate": "{:.1%}".format,
    "roc_auc": "{:.3f}".format,
    "average_precision": "{:.3f}".format,
    "precision_at_20": "{:.1%}".format,
    "precision_at_50": "{:.1%}".format,
    "precision_at_100": "{:.1%}".format,
}))
print(
    f"\nRandom Forest P@50 / test base rate: "
    f"{model_metrics['precision_at_50'] / test_base_rate:.2f}x"
)

feature_importance_table = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "importance": model.named_steps["classifier"].feature_importances_,
    }
).sort_values("importance", ascending=False)
print("\nTop model features (impurity importance; descriptive, not causal):")
print(feature_importance_table.head(10).to_string(index=False, formatters={"importance": "{:.3f}".format}))

monthly_rates = panel.groupby("target_month")["is_declining"].mean().sort_index()
print("\nObserved decline-label rate by target month (coverage-filtered examples):")
print(monthly_rates.map(lambda value: f"{value:.1%}").to_string())

               Ranker base_rate roc_auc average_precision precision_at_20 precision_at_50 precision_at_100
Recent-trend baseline     41.0%   0.721             0.646           90.0%           90.0%            86.0%
        Random Forest     41.0%   0.720             0.646           90.0%           90.0%            87.0%

Random Forest P@50 / test base rate: 2.19x

Top model features (impurity importance; descriptive, not causal):
                     feature importance
historical_impression_change      0.549
             position_change      0.127
    log_previous_impressions      0.095
                 current_ctr      0.065
     log_current_impressions      0.040
                previous_ctr      0.032
          log_current_clicks      0.030
        current_avg_position      0.025
       previous_avg_position      0.020
         log_previous_clicks      0.016

Observed decline-label rate by target month (coverage-filtered examples):
target_month
2025-04-01     8.8%
2025-05-01    10.8%

## 5. Limitations and honest framing

The test is one future month and a seeded sample of clients, so it is evidence about this release and this evaluation design - not proof of performance across every client, season, or later period. The same content can appear in earlier training months for non-held-out clients; the June test clients are fully excluded from training. The 20-day and 200-impression filters select measurable, sufficiently visible items and exclude low-volume or data-sparse pages.

The target is a 20% relative decrease in a monthly GSC rate, not a human judgment of content quality. Search demand, seasonality, technical changes, measurement coverage, and ranking changes can all affect impressions. The study is observational and cannot show that refreshing content causes recovery. Random Forest scores are ranking scores, not calibrated probabilities of an editorial action succeeding. No revenue, editorial cost, or intervention outcomes are measured.

In [16]:
forbidden_model_inputs = {
    "client_hash_id", "content_hash_id", "is_declining", "target_month",
    "target_impressions_per_gsc_day", "trend_direction", "trend_pct",
}
forbidden_present = sorted(forbidden_model_inputs.intersection(FEATURE_COLUMNS))
if forbidden_present:
    raise AssertionError(f"Identifier or target-source leakage in features: {forbidden_present}")
if not train_df["target_month"].lt(TEST_TARGET_MONTH).all():
    raise AssertionError("A training label is not strictly earlier than the test month.")
if not test_df["target_month"].eq(TEST_TARGET_MONTH).all():
    raise AssertionError("The test set contains a target month other than June 2026.")
if not train_clients.isdisjoint(test_clients):
    raise AssertionError("A client appears in both training and test groups.")
if not panel["target_month"].eq(panel["feature_month"] + pd.offsets.MonthBegin(1)).all():
    raise AssertionError("A target month is not the month immediately after its feature month.")
print("Leakage/time checks passed: IDs and future labels are not model features; train/test clients are disjoint.")
print("All feature months precede their target months, and training target months precede June 2026.")
print("Limitations to retain in the paper:")
for limitation in [
    "One release and one future test month; no claim about later periods.",
    "A client holdout supports transfer to unseen clients, not universal client coverage.",
    "Minimum coverage/volume filters exclude sparse and low-impression pages.",
    "No causal effect, calibrated intervention probability, revenue, or editorial-cost estimate.",
]:
    print("-", limitation)

Leakage/time checks passed: IDs and future labels are not model features; train/test clients are disjoint.
All feature months precede their target months, and training target months precede June 2026.
Limitations to retain in the paper:
- One release and one future test month; no claim about later periods.
- A client holdout supports transfer to unseen clients, not universal client coverage.
- Minimum coverage/volume filters exclude sparse and low-impression pages.
- No causal effect, calibrated intervention probability, revenue, or editorial-cost estimate.


## 6. Ranked recommendations - a human review playbook

The ranked list is a **retrospective evaluation queue** for held-out clients, built from April-May 2026 signals to assess June outcomes. It is not a live queue: June outcomes are already present in the release. In a live workflow, rebuild the same features only through the latest complete month and score the next month.

Prioritize the highest-ranked items for a manual diagnostic. Use the recent impression trend and position change as reason codes, then check seasonality, indexing/technical status, search-intent fit, and whether the page was already changed. Treat a high model score without a clear recent signal as a prompt to investigate, not an instruction to rewrite. Never auto-publish, redirect, merge, or delete content based on this score.

In [17]:
def review_route(row):
    recent_change = row["historical_impression_change"]
    position_change = row["position_change"]
    if pd.notna(recent_change) and recent_change <= -0.20 and pd.notna(position_change) and position_change >= 1.0:
        return (
            "recent_demand_and_position_drop",
            "Check seasonality, technical/indexing health, and search results before editing.",
        )
    if pd.notna(recent_change) and recent_change <= -0.20:
        return (
            "recent_impression_drop",
            "Verify the trend and inspect demand/intent context before editing.",
        )
    if pd.notna(position_change) and position_change >= 1.0:
        return (
            "position_worsening",
            "Review query intent and SERP context; do not assume content is the cause.",
        )
    return (
        "model_risk_needs_context",
        "Check measurement and page context; the model signal alone is insufficient.",
    )

routes = test_scored.apply(review_route, axis=1, result_type="expand")
test_scored["reason_code"] = routes[0]
test_scored["suggested_review"] = routes[1]
ranked_queue = test_scored.sort_values("model_score", ascending=False, kind="mergesort").copy()
ranked_queue.insert(0, "queue_rank", np.arange(1, len(ranked_queue) + 1))
top50_queue = ranked_queue.head(min(50, len(ranked_queue))).copy()

queue_path = work_outputs / "capstone_ranked_queue.csv"
queue_columns = [
    "queue_rank", "client_hash_id", "content_hash_id", "target_month",
    "model_score", "baseline_score", "reason_code", "suggested_review",
]
ranked_queue[queue_columns].to_csv(queue_path, index=False)
recommendation_counts = top50_queue["suggested_review"].value_counts().sort_values(ascending=False)
reason_counts_top50 = top50_queue["reason_code"].value_counts().sort_values(ascending=False)
print("Top-50 recommended review mix (aggregate only):")
print(recommendation_counts.to_string())
print("\nTop-50 reason-code mix (aggregate only):")
print(reason_counts_top50.to_string())
print(f"\nLocal ranked queue saved to {queue_path}; it is gitignored and must not be committed.")
print("No row-level IDs or target labels are displayed or included in the public report.")

Top-50 recommended review mix (aggregate only):
suggested_review
Check seasonality, technical/indexing health, and search results before editing.    49
Verify the trend and inspect demand/intent context before editing.                   1

Top-50 reason-code mix (aggregate only):
reason_code
recent_demand_and_position_drop    49
recent_impression_drop              1

Local ranked queue saved to d:\git\Assignment-1\work\outputs\capstone_ranked_queue.csv; it is gitignored and must not be committed.
No row-level IDs or target labels are displayed or included in the public report.


## 7. Artifacts the paper embeds

The code below creates aggregate, public-safe SVGs and a metrics receipt from the full-release evaluation. Each chart has a clear unit and title; the report should place a one-sentence takeaway below each figure. The row-level review queue and the cached feature panel stay local and gitignored.

- `work/figures/capstone_precision_at_k.svg` - baseline/model P@20 and P@50, with the held-out test base rate.
- `work/figures/capstone_monthly_decline_rate.svg` - measured target rate by month after coverage filtering.
- `work/figures/capstone_feature_importance.svg` - top Random Forest importance values, not causal effects.
- `work/figures/capstone_top50_recommendation_mix.svg` - aggregate review routes in the top 50.
- `work/outputs/capstone_metrics_receipt.json` - aggregate metrics and method settings only.

In [18]:
import json
import matplotlib.pyplot as plt

work_figures = repo_root / "work" / "figures"
work_figures.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.titlesize": 13, "axes.labelsize": 10})

fig, ax = plt.subplots(figsize=(7.4, 4.4))
ks = ["P@20", "P@50"]
positions = np.arange(len(ks))
width = 0.34
baseline_values = [baseline_metrics["precision_at_20"], baseline_metrics["precision_at_50"]]
model_values = [model_metrics["precision_at_20"], model_metrics["precision_at_50"]]
bars1 = ax.bar(positions - width / 2, baseline_values, width, label="Recent-trend baseline", color="#7b8fa1")
bars2 = ax.bar(positions + width / 2, model_values, width, label="Random Forest", color="#2b756e")
ax.axhline(test_base_rate, color="#a44a3f", linestyle="--", linewidth=1.5, label="Test decline base rate")
ax.set_xticks(positions, ks)
ax.set_ylim(0, 1)
ax.set_ylabel("Fraction of top-ranked items with a future decline")
ax.set_title("Ranking quality on June 2026 held-out clients")
ax.legend(frameon=False, loc="upper right")
ax.bar_label(bars1, fmt="%.1%%", padding=2, fontsize=9)
ax.bar_label(bars2, fmt="%.1%%", padding=2, fontsize=9)
fig.tight_layout()
precision_figure = work_figures / "capstone_precision_at_k.svg"
fig.savefig(precision_figure, format="svg", bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 4.4))
month_labels = monthly_rates.index.strftime("%b %Y")
ax.plot(month_labels, monthly_rates.values, marker="o", color="#2b756e", linewidth=2)
ax.axvline(len(month_labels) - 1, color="#a44a3f", linestyle="--", linewidth=1.2)
ax.set_ylim(0, 1)
ax.set_ylabel("Share labeled as >20% decline")
ax.set_title("Observed decline rate by target month")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
monthly_figure = work_figures / "capstone_monthly_decline_rate.svg"
fig.savefig(monthly_figure, format="svg", bbox_inches="tight")
plt.close(fig)

importance_plot = feature_importance_table.head(10).sort_values("importance", ascending=True)
fig, ax = plt.subplots(figsize=(8, 5.2))
ax.barh(importance_plot["feature"], importance_plot["importance"], color="#6f4e7c")
ax.set_xlabel("Random Forest impurity importance")
ax.set_title("Top model features (descriptive, not causal)")
fig.tight_layout()
importance_figure = work_figures / "capstone_feature_importance.svg"
fig.savefig(importance_figure, format="svg", bbox_inches="tight")
plt.close(fig)

action_plot = recommendation_counts.sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(action_plot.index, action_plot.values, color="#426b69")
ax.set_xlabel("Items in model-ranked top 50")
ax.set_title("Aggregate human-review routes")
fig.tight_layout()
recommendation_figure = work_figures / "capstone_top50_recommendation_mix.svg"
fig.savefig(recommendation_figure, format="svg", bbox_inches="tight")
plt.close(fig)

metrics_receipt = {
    "dataset": {
        "repository": "FlyRank/internship-warehouse",
        "build": "flyrank_pseudonymized_warehouse_release_v20260703",
        "table": "fact_content_daily_performance",
        "published_rows": 78835655,
        "report_date_min": "2025-01-27",
        "report_date_max": "2026-06-30",
    },
    "analysis": {
        "eligible_page_month_examples": int(len(panel)),
        "minimum_gsc_days_per_month": 20,
        "minimum_prior_two_month_impressions": 200,
        "target": "next-month average daily impressions < 80% of the mean in the prior two months",
        "feature_columns": FEATURE_COLUMNS,
        "test_target_month": TEST_TARGET_MONTH.strftime("%Y-%m"),
        "train_target_month_min": train_df["target_month"].min().strftime("%Y-%m"),
        "train_target_month_max": train_df["target_month"].max().strftime("%Y-%m"),
        "train_examples": int(len(train_df)),
        "test_examples": int(len(test_df)),
        "held_out_clients_count": int(test_client_count),
        "random_state": SEED,
    },
    "test_decline_base_rate": test_base_rate,
    "ranking_metrics": {
        "recent_trend_baseline": baseline_metrics,
        "random_forest": model_metrics,
    },
    "top_feature_importance": [
        {"feature": str(row.feature), "importance": float(row.importance)}
        for row in feature_importance_table.head(10).itertuples(index=False)
    ],
    "top50_review_counts": {str(key): int(value) for key, value in recommendation_counts.items()},
    "top50_reason_counts": {str(key): int(value) for key, value in reason_counts_top50.items()},
    "figures": [
        "work/figures/capstone_precision_at_k.svg",
        "work/figures/capstone_monthly_decline_rate.svg",
        "work/figures/capstone_feature_importance.svg",
        "work/figures/capstone_top50_recommendation_mix.svg",
    ],
    "publication_note": "Aggregate results only; the hashed-ID queue and feature cache remain local and uncommitted.",
}
receipt_path = work_outputs / "capstone_metrics_receipt.json"
receipt_path.write_text(json.dumps(metrics_receipt, indent=2, allow_nan=False), encoding="utf-8")
print("Paper-ready aggregate figures:")
for path in [precision_figure, monthly_figure, importance_figure, recommendation_figure]:
    print("-", path.relative_to(repo_root).as_posix())
print("Aggregate metrics receipt:", receipt_path.relative_to(repo_root).as_posix())

Paper-ready aggregate figures:
- work/figures/capstone_precision_at_k.svg
- work/figures/capstone_monthly_decline_rate.svg
- work/figures/capstone_feature_importance.svg
- work/figures/capstone_top50_recommendation_mix.svg
Aggregate metrics receipt: work/outputs/capstone_metrics_receipt.json


## 8. Reproducibility

From a fresh clone, install the project dependencies, request and accept the Hugging Face dataset gate, authenticate with a **Read** token using the local CLI, then run this notebook from top to bottom. The notebook reads credentials from the local Hugging Face cache or `HF_TOKEN`; it never writes a token into the notebook or repository. The final monthly feature panel is cached as an ignored Parquet file so rerunning the notebook does not rescan the 78.8M-row fact table. Delete that cache only when intentionally rebuilding the analysis from the same release. The split seed and model seed are both 42; model parameters are recorded in the code and metrics receipt.

In [19]:
import duckdb
import sklearn
import sys

reproduction_steps = [
    "python -m pip install -r requirements.txt",
    "hf auth login  # Read token entered at the terminal prompt; never paste it into a notebook",
    "Open work/notebooks/capstone.ipynb and Run All",
    "Optional full refresh: delete work/outputs/capstone_feature_panel_v1.parquet, then Run All once",
    "After deployment: write the exact public URL to submission/paper_url.txt (one line)",
]
print("Reproduction steps:")
for step in reproduction_steps:
    print("-", step)
print(f"\nPython: {sys.version.split()[0]}; DuckDB: {duckdb.__version__}; pandas: {pd.__version__}; scikit-learn: {sklearn.__version__}")
print(f"Random state: {SEED}; feature cache exists: {feature_cache.is_file()}")

Reproduction steps:
- python -m pip install -r requirements.txt
- hf auth login  # Read token entered at the terminal prompt; never paste it into a notebook
- Open work/notebooks/capstone.ipynb and Run All
- Optional full refresh: delete work/outputs/capstone_feature_panel_v1.parquet, then Run All once
- After deployment: write the exact public URL to submission/paper_url.txt (one line)

Python: 3.12.3; DuckDB: 1.5.5; pandas: 3.0.5; scikit-learn: 1.9.0
Random state: 42; feature cache exists: True


## 9. Acknowledgments and data credit

This work was built on the **FlyRank ML Internship warehouse release** (build `flyrank_pseudonymized_warehouse_release_v20260703`), sourced from `central_data_warehouse`. Data credit: [FlyRank](https://flyrank.ai) and the [FlyRank internship warehouse release](https://huggingface.co/datasets/FlyRank/internship-warehouse). The release is pseudonymized; this paper reports aggregates and does not include client names, page URLs, titles, keywords, or raw queries. The analysis notebooks and code are available in the [Assignment-1 repository](https://github.com/MahdiaFaheem/Assignment-1).

In [20]:
print("Data credit: FlyRank ML Internship warehouse, build v20260703.")
print("Source page: https://huggingface.co/datasets/FlyRank/internship-warehouse")
print("Repository: https://github.com/MahdiaFaheem/Assignment-1")
print("Public outputs are aggregate only; the local hashed-ID queue and feature cache are not submitted.")

Data credit: FlyRank ML Internship warehouse, build v20260703.
Source page: https://huggingface.co/datasets/FlyRank/internship-warehouse
Repository: https://github.com/MahdiaFaheem/Assignment-1
Public outputs are aggregate only; the local hashed-ID queue and feature cache are not submitted.


## Self-check

Before submitting, confirm each line honestly:

- [ ] All required paper sections are complete, including the five-sentence abstract and data credit
- [ ] Notebook executed top to bottom without errors using the full warehouse release
- [ ] Model and baseline use the same future-month/client-held-out test; base rate is shown
- [ ] Leakage, coverage, and causal-claim limitations are stated
- [ ] No client names, URLs, titles, keywords, private queries, or row-level outputs appear in the public paper
- [ ] Paper figures and aggregate metrics receipt are included; local Parquet/queue files stay uncommitted
- [ ] Deployed paper is public and the exact URL is in `submission/paper_url.txt`
- [ ] Notebook/report changes are committed and the repo URL is submitted on the card